In [27]:
from utils import *
from heuristique_glouton import *
from heuristique_itérative import *
from evaluation import *

In [28]:
df_ville,df_object,capacity=parse_ttp_file("a280_n279_bounded-strongly-corr_01.ttp")

In [29]:
df_ville

,X,Y
1,288,149
2,288,129
3,270,133
4,256,141
5,256,157
...,...,...
276,236,145
277,246,141
278,252,125
279,260,129


In [30]:
df_object

,Profit,Weight,City_Index
1,101,1,2
2,202,2,3
3,404,4,4
4,202,2,5
5,996,896,6
...,...,...,...
275,786,686,276
276,1572,1372,277
277,786,686,278
278,566,466,279


In [31]:
pi,obj_pris,poids_tot,dict_ville_objet_pris=algo_glouton(df_ville,df_object,capacity)

Loading:  96%|█████████▌| 24949/25936 [00:12<00:00, 1993.20weight unit/s, Current Weight=24949, Nb_ville=254]


In [32]:
for ville in pi:
    print("Dans la ville ",ville," le voleur prend ",len(dict_ville_objet_pris[ville])," objet(s).")

Dans la ville  1  le voleur prend  0  objet(s).
Dans la ville  6  le voleur prend  0  objet(s).
Dans la ville  7  le voleur prend  0  objet(s).
Dans la ville  178  le voleur prend  0  objet(s).
Dans la ville  99  le voleur prend  0  objet(s).
Dans la ville  96  le voleur prend  0  objet(s).
Dans la ville  97  le voleur prend  0  objet(s).
Dans la ville  98  le voleur prend  0  objet(s).
Dans la ville  93  le voleur prend  0  objet(s).
Dans la ville  94  le voleur prend  0  objet(s).
Dans la ville  95  le voleur prend  0  objet(s).
Dans la ville  76  le voleur prend  0  objet(s).
Dans la ville  117  le voleur prend  0  objet(s).
Dans la ville  115  le voleur prend  0  objet(s).
Dans la ville  116  le voleur prend  0  objet(s).
Dans la ville  64  le voleur prend  0  objet(s).
Dans la ville  56  le voleur prend  0  objet(s).
Dans la ville  55  le voleur prend  0  objet(s).
Dans la ville  54  le voleur prend  0  objet(s).
Dans la ville  53  le voleur prend  0  objet(s).
Dans la ville  38  

In [33]:
eval_non_lin(pi,df_ville,df_object,dict_ville_objet_pris,obj_pris,capacity)

yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
BENEFICE :  39249
COUT :  6840.9321285460655


(32408.067871453935, 39249, 6840.9321285460655)

In [34]:
eval_lin(pi,df_ville,df_object,dict_ville_objet_pris,obj_pris)

yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
BENEFICE :  39249
COUT :  31281285.950658288


(-31242036.950658288, 39249, 31281285.950658288)

In [35]:
import gurobipy as gp

# Initialize combined model
model = gp.Model("CombinedModel")

# Decision variables for object selection
dict_obj_ville = {}
list_index_ville = list(df_ville.index)
list_obj = []

for index_ville in list_index_ville:
    list_index_obj = list(get_objects_of_ville(index_ville + 1, df_object).index)
    dict_obj_ville[index_ville] = []
    for index_obj in list_index_obj:
        x = model.addVar(vtype=gp.GRB.BINARY, name=f"x_{index_ville}_{index_obj}")
        dict_obj_ville[index_ville].append(x)
        list_obj.append(x)


# Decision variables for routing
n = len(list_index_ville)
y = [
    [model.addVar(vtype=gp.GRB.BINARY, name=f"y_{i}_{j}") for j in range(n)]
    for i in range(n)
]

# Auxiliary variables for cumulative weight
w = model.addVars(n, vtype=gp.GRB.CONTINUOUS, lb=0, name="w")

w_transfers=model.addVars(n, vtype=gp.GRB.CONTINUOUS, lb=0, name="w_transfer")

model.addConstr(w_transfers[0]==0)

# Auxiliary variables for MTZ constraints
u = model.addVars(n, vtype=gp.GRB.CONTINUOUS, lb=1, ub=n, name="u")

# Define weights and profits for object selection
list_poids = [df_object.iloc[i]["Weight"] for i in range(len(list_obj))]
list_benefits = [df_object.iloc[i]["Profit"] for i in range(len(list_obj))]

# Define distance matrix
matrix_distance = {i: calcul_distance_de_ville(i, df_ville) for i in list_index_ville}

# Add capacity constraint
model.addConstr(
    gp.quicksum(list_poids[i] * list_obj[i] for i in range(len(list_poids))) <= capacity,
    "Capacity"
)

# Add routing constraints
model.addConstrs(
    (gp.quicksum(y[i][j] for j in range(n) if j != i) == 1 for i in range(n)), "Depart"
)
model.addConstrs(
    (gp.quicksum(y[i][j] for i in range(n) if i != j) == 1 for j in range(n)), "Arrive"
)

# Ensure objects are selected only if their cities are visited
for index_ville, vars_list in dict_obj_ville.items():
    for obj_var in vars_list:
        model.addConstr(
            gp.quicksum(y[index_ville][j] for j in range(n) if j != index_ville) >= obj_var,
            f"ObjectSelectedIfCityVisited_{index_ville}"
        )

# Add cumulative weight constraints
for i in range(n):
    model.addConstr(
        w[i] == gp.quicksum(dict_obj_ville[i+1][k] * list_poids[k] for k in range(len(dict_obj_ville[i+1]))),
        f"CumulativeWeight_{i}"
    )
    if i > 0:
        model.addConstr(w_transfers[i] == w[i] + w_transfers[i-1],f"WeightTransfer_{i}")

# Add MTZ constraints to eliminate subtours
for i in range(1, n):  # Start from 1 since city 0 is the starting point
    for j in range(1, n):  # MTZ does not apply for starting city
        if i != j:
            model.addConstr(
                u[i] - u[j] + n * y[i][j] <= n - 1,
                name=f"SubtourElimination_{i}_{j}"
            )

# Define objective function
profit = gp.quicksum(list_benefits[i] * list_obj[i] for i in range(len(list_benefits)))
routing_cost = gp.quicksum(
    matrix_distance[i + 1][j + 1] * y[i][j] * w[i] for i in range(n) for j in range(n)
)

# Combine objectives: maximize profit and minimize routing cost
alpha = 1  # Weight for profit
beta = 1   # Weight for routing cost (adjust based on importance)
model.setObjective(alpha * profit-beta*routing_cost, gp.GRB.MAXIMIZE)

# Optimize the model
model.optimize()

# Print results
if model.status == gp.GRB.OPTIMAL:
    print(f"Optimal combined objective value: {model.objVal}")
    print("Selected objects:")
    for index_ville, vars_list in dict_obj_ville.items():
        for obj_var in vars_list:
            if obj_var.x > 0.5:
                print(f"Object {obj_var.varName} selected")
    print("Optimal routing:")
    for i in range(n):
        for j in range(n):
            if y[i][j].x > 0.5:
                print(f"Route: City {i} -> City {j}")


Gurobi Optimizer version 12.0.0 build v12.0.0rc1 (win64 - Windows 10.0 (19045.2))

CPU model: Intel(R) Core(TM) i7-1065G7 CPU @ 1.30GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 78962 rows, 79519 columns and 468722 nonzeros
Model fingerprint: 0xdccd3911
Model has 78118 quadratic objective terms
Variable types: 840 continuous, 78679 integer (78679 binary)
Coefficient statistics:
  Matrix range     [1e+00, 4e+03]
  Objective range  [1e+02, 4e+03]
  QObjective range [2e+01, 6e+02]
  Bounds range     [1e+00, 3e+02]
  RHS range        [1e+00, 3e+04]
Presolve removed 839 rows and 841 columns
Presolve time: 1.00s
Presolved: 155962 rows, 156517 columns, 622722 nonzeros
Variable types: 279 continuous, 156238 integer (156238 binary)
Deterministic concurrent LP optimizer: primal simplex, dual simplex, and barrier
Showing barrier log only...

Root barrier log...

Ordering time: 0.08s

Barrier statistic

In [36]:
print("Capacity:", capacity)
print("Weights:", list_poids)
print("Profits:", list_benefits)
print("Distance matrix:", matrix_distance)

Capacity: 25936
Weights: [1, 2, 4, 2, 896, 1792, 3584, 367, 734, 1468, 690, 690, 613, 1226, 2452, 874, 874, 122, 244, 488, 366, 486, 972, 972, 823, 823, 463, 926, 463, 589, 1178, 1178, 325, 650, 1300, 975, 1000, 2000, 4000, 776, 1552, 2328, 123, 356, 305, 610, 1220, 305, 47, 94, 188, 645, 1290, 2580, 1290, 139, 247, 494, 988, 741, 211, 422, 844, 633, 100, 200, 400, 100, 627, 1254, 1254, 760, 1520, 776, 1552, 3104, 475, 950, 1900, 937, 1874, 3748, 937, 101, 202, 404, 202, 281, 562, 1124, 562, 751, 1502, 3004, 1502, 530, 1060, 2120, 240, 480, 480, 248, 496, 744, 768, 31, 62, 31, 613, 1226, 2452, 613, 881, 1762, 1762, 367, 734, 1468, 367, 712, 410, 820, 1640, 410, 157, 314, 628, 471, 824, 242, 472, 944, 1416, 722, 1444, 2888, 775, 775, 371, 742, 1484, 337, 674, 337, 885, 1770, 3540, 1770, 204, 408, 816, 408, 594, 1188, 2376, 1188, 226, 452, 904, 226, 452, 921, 537, 1074, 2148, 112, 112, 959, 1918, 2877, 392, 392, 709, 1418, 2836, 940, 1880, 943, 1886, 3772, 79, 79, 922, 1844, 1844, 361, 9